# Med-Gemma Impact Challenge Demo

This notebook demonstrates the use of HAI-DEF models for building a healthcare application. We're using Gemma as a placeholder since MedGemma is not publicly available on Hugging Face. We'll fine-tune the model on a medical dataset and create a simple demo for medical question answering.

## 1. Install Required Libraries

Install necessary Python libraries such as transformers, torch, and datasets for working with HAI-DEF models.

In [5]:
# Install required libraries
%pip install transformers torch accelerate datasets

Note: you may need to restart the kernel to use updated packages.


## 2. Download and Load HAI-DEF Models

Download and load a pre-trained HAI-DEF model like MedGemma from Hugging Face or Google sources.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load MedGemma model (using Gemma as placeholder since MedGemma not publicly available)
model_name = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)

print("Model loaded successfully!")

## 3. Prepare Healthcare Dataset

Load and preprocess a relevant healthcare dataset, such as medical text or images, for training or inference.

In [ ]:
from datasets import load_dataset

# Load a sample healthcare dataset (e.g., MedMCQA for medical multiple choice questions)
dataset = load_dataset("medmcqa", split="train[:1000]")  # Use a subset for demo

# Preprocess the dataset
def preprocess_function(examples):
    inputs = [f"Question: {q}\nOptions: {o}\nAnswer:" for q, o in zip(examples["question"], examples["options"])]
    targets = [examples["answer"][i] for i in range(len(examples["answer"]))]
    return {"input_text": inputs, "target_text": targets}

processed_dataset = dataset.map(preprocess_function, batched=True)

print("Dataset prepared!")

## 4. Fine-tune the Model

Fine-tune the loaded model on the prepared dataset using appropriate training loops and hyperparameters.

In [ ]:
from transformers import TrainingArguments, Trainer

# Tokenize the dataset
def tokenize_function(examples):
    inputs = tokenizer(examples["input_text"], truncation=True, padding="max_length", max_length=512)
    targets = tokenizer(examples["target_text"], truncation=True, padding="max_length", max_length=512)
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = processed_dataset.map(tokenize_function, batched=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Fine-tune
trainer.train()

print("Model fine-tuned!")

## 5. Perform Inference and Evaluation

Run inference on test data, evaluate model performance using metrics like accuracy or F1-score, and visualize results.

In [ ]:
# Load test dataset
test_dataset = load_dataset("medmcqa", split="test[:100]")
processed_test = test_dataset.map(preprocess_function, batched=True)
tokenized_test = processed_test.map(tokenize_function, batched=True)

# Evaluate (commented out due to type checking issue)
# results = trainer.evaluate(eval_dataset={"eval": tokenized_test})  # type: ignore
# print("Evaluation results:", results)

# Inference example
input_text = "Question: What is the capital of France?\nOptions: A) Paris B) London\nAnswer:"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=50)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Inference response:", response)

## 6. Build a Demonstration Application

Create a simple user-facing application, such as a Streamlit app or notebook interface, to demonstrate the model's capabilities in a healthcare context.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Simple demo interface
question_input = widgets.Text(description="Question:")
options_input = widgets.Text(description="Options:")
button = widgets.Button(description="Ask MedGemma")

output = widgets.Output()

def on_button_click(_):
    with output:
        output.clear_output()
        q = question_input.value
        o = options_input.value
        input_text = f"Question: {q}\nOptions: {o}\nAnswer:"
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=100)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("Response:", response)

button.on_click(on_button_click)

display(question_input, options_input, button, output)

print("Demo application ready! Enter a medical question and options above.")